# S2 Pair Panel (1H) — only if H-002 freezes 1H

**Do not run this unless `BAR_STAR == "1h"`.** Universe C has **no 4H** panel.

Rebuilds `s2_panel_C_1h_{train,full}.parquet` from Yahoo 1H using session-scaled lookbacks
(`252/60/252` days × 6 bars/session). Locked pairs only. 70/30 IS:OOS on the 1H window.


## 0. Imports & Config


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.report import (
    fold_table,
    fold_val_metrics,
    load_star_stack,
    median_sharpe_hint,
    plot_fold_boxplots,
    require_star,
    save_star_stack,
    write_tearsheet_pdf,
)
from backtest.s2_coint.research import (
    ARTIFACTS_DIR,
    DEFAULT_STAR_STACK,
    config_from_stack,
    is_end_for_stack,
    load_s1_weekly,
    load_universe_c_panels,
    lookbacks_for_bar,
    overlay_kalman_hedge,
    repo_root,
    split_is_oos,
    tearsheet_path,
)
from backtest.s2_coint.runner import run_s2_backtest
from backtest.s2_coint.walkforward import embargo_bars_for_config, make_s2_folds
from strategies.s2_coint.config import S2SimConfig

STAR_PATH = DEFAULT_STAR_STACK
TEARSHEET_DIR = ARTIFACTS_DIR
stack = load_star_stack(STAR_PATH)
PAIRS_STAR = list(stack["PAIRS_STAR"])
print("stack keys:", sorted(stack))
print("PAIRS_STAR", PAIRS_STAR)


## 1. Fetch 1H and build panel


In [ ]:
from data.ingestion.equity_fetcher import fetch_ohlcv
from data.processing.s2_coint_store import build_pair_panel
from backtest.s2_coint.research import long_ohlcv_to_frames, overlap_is_end, pair_tuples, unique_tickers

require_star("BAR_STAR", stack.get("BAR_STAR"))
assert stack["BAR_STAR"] == "1h", "this notebook is only for the 1H sample rule"
lb = lookbacks_for_bar("1h")
frames = {}
for t in unique_tickers(PAIRS_STAR):
    frames.update(long_ohlcv_to_frames(fetch_ohlcv(t, "2018-01-01", interval="1h", isAsian=True)))
full = build_pair_panel(
    frames,
    pair_tuples(PAIRS_STAR),
    ols_window=lb["ols_window"],
    z_window=lb["z_window"],
    hl_window=lb["hl_window"],
    include_adf_pvalue=True,
    include_variance_jump=True,
)
is_end = overlap_is_end(full, frac=0.70)
train, _ = split_is_oos(full, is_end=is_end)
data_dir = os.path.join(ROOT, "01_data", "data_files", "s2_coint")
os.makedirs(data_dir, exist_ok=True)
train.to_parquet(os.path.join(data_dir, "s2_panel_C_1h_train.parquet"), index=False)
full.to_parquet(os.path.join(data_dir, "s2_panel_C_1h_full.parquet"), index=False)
print("wrote 1h train/full", len(train), len(full), "is_end", is_end)
